# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Step 1: Load Creator Open IDs

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
list_all_creators = df_creators_list['username'].tolist()

In [5]:
# Load or initialize the manifest (tracks which handles are already found)
manifest = pd.read_csv(MANIFEST_CSV)
df_creators = pd.read_csv(CONSOLIDATED_CSV)

# recheck manifest in case of failed runs
manifest["found"] = manifest["handle"].isin(set(df_creators['username']))
manifest.to_csv(MANIFEST_CSV, index=False)

In [6]:
manifest['found'].sum()

np.int64(3159)

# Step 2: Extract List of existing target collaborations to avoid invitation conflicts

In [7]:
# load IDs to check
creator_open_id_list = df_creators.loc[df_creators['username'].isin(set(manifest['handle'])), 'creator_open_id'].tolist()
product_id_list = ['1734810555690551128']

In [8]:
# Load known conflicts from previous runs
conflicts_manifest_df = pd.read_csv(CONFLICTS_MANIFEST_CSV, dtype=str)

known_conflict_ids = set(conflicts_manifest_df["creator_open_id"])

# Skip creators already known to conflict
creators_to_check = [c for c in creator_open_id_list if c not in known_conflict_ids]

skipped_count = len(creator_open_id_list) - len(creators_to_check)
if skipped_count:
    print(f"Skipping {skipped_count} creator(s) already known to conflict. Checking {len(creators_to_check)} potentially new creators.")

Skipping 2639 creator(s) already known to conflict. Checking 520 potentially new creators.


In [9]:
# find all open IDs with conflicts
all_conflict_items = []

for i in range(0, len(creators_to_check), 50):
    batch = creators_to_check[i:i + 50]
    result = check_target_collaboration_conflicts(
        creator_open_id_list=batch,
        product_id_list=product_id_list,
    )

    if result.get("code") == 0:
        all_conflict_items.extend(result["data"]["conflict_items"])
    else:
        print(f"  ⚠️  Batch failed: {result}")
        print(batch)
        
df_all_collab_conflicts = pd.DataFrame(all_conflict_items)

# Merge any newly found conflicts into the manifest
if not df_all_collab_conflicts.empty:
    df_all_collab_conflicts = pd.concat([conflicts_manifest_df, df_all_collab_conflicts], ignore_index=True)
    df_all_collab_conflicts = df_all_collab_conflicts.drop_duplicates(subset=["creator_open_id"], keep="first")
    df_all_collab_conflicts.to_csv(CONFLICTS_MANIFEST_CSV, index=False)
    new_count = len(df_all_collab_conflicts) - len(conflicts_manifest_df)
    print(f"Manifest now has {len(df_all_collab_conflicts)} known conflict(s) ({new_count} new this run).")
else:
    df_all_collab_conflicts = conflicts_manifest_df

Manifest now has 2800 known conflict(s) (161 new this run).


In [10]:
df_all_collab_conflicts

,creator_open_id,existing_collaboration_id,product_id
0,sQWUmQAAAACOV4AEeERJl-yievH_HBssk7035z_rR3rfGg...,7661970332203550472,1734810555690551128
1,XoTXawAAAACOV4AEeERJl-yievH_HBssONvLAt04stNsY9...,7661970332203550472,1734810555690551128
2,fTcLggAAAACOV4AEeERJl-yievH_HBssZ2wBpc8HcDmwra...,7661970334190913301,1734810555690551128
3,sW6t0wAAAACOV4AEeERJl-yievH_HBssY7WK-VuIXdWMLi...,7661970369088603924,1734810555690551128
4,tKpu7wAAAACOV4AEeERJl-yievH_HBssgYMxHyHuweMR6j...,7661970518602401552,1734810555690551128
...,...,...,...
2795,zFZ5vAAAAACOV4AEeERJl-yievH_HBssmqnBJKsEFDtYrH...,7660461201453172498,1734810555690551128
2796,HHdpmwAAAACOV4AEeERJl-yievH_HBsshMb1rg_loKv5ML...,7660461201453172498,1734810555690551128
2797,-0g3qAAAAACOV4AEeERJl-yievH_HBssi_lzqQAT6LR79e...,7657956260486792967,1734810555690551128
2798,v6xJLwAAAACOV4AEeERJl-yievH_HBsshopfT_E_Q1_iPu...,7658273774026311432,1734810555690551128


## Review current status

In [11]:
# merge all extracted data
df_creators_list_id_conflict = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_all_collab_conflicts, how='left', on="creator_open_id")

In [12]:
df_creator_summary = df_creators_list_id_conflict.groupby('batch_name').agg(
    all_creators=('creator_open_id', 'size'),
    creators_with_id=('creator_open_id', 'count'),
    invited=('existing_collaboration_id', 'count'),
)

df_creator_summary['to_invite'] = df_creator_summary['creators_with_id'] - df_creator_summary['invited']

In [13]:
df_creator_summary

,all_creators,creators_with_id,invited,to_invite
batch_name,,,,
All_202606_00k-02k,1395,1010,950,60
All_202606_02k-04k,1534,47,21,26
All_202606_04k-06k,1630,41,10,31
All_202606_06k-08k,1660,36,12,24
All_202606_08k-10k,1672,19,8,11
All_202606_10k-12k,1707,17,2,15
Health_202606_00k-02k,2000,1688,1546,142
Health_202606_02k-04k,2000,226,214,12
Health_202606_04k-06k,2000,23,11,12


## Prepare shortlisted file without conflicts for batch processing

In [14]:
# set aside all new creators with open IDs but no conflicts
df_creators_list_id_new = df_creators_list_id_conflict[df_creators_list_id_conflict['creator_open_id'].notnull() & df_creators_list_id_conflict['existing_collaboration_id'].isnull()].reset_index(drop=True).copy()

In [15]:
# check counts
num_creators_with_conflict = df_creators_list_id_conflict['existing_collaboration_id'].notnull().sum()
num_creators_with_id = df_creators_list_id_conflict['creator_open_id'].notnull().sum()

print(f"{num_creators_with_conflict} out of {num_creators_with_id} with conflicts. {df_creators_list_id_new.shape[0]} new creator_open_id remaining for new target collaborations")

2800 out of 3159 with conflicts. 359 new creator_open_id remaining for new target collaborations


In [16]:
df_creators_list_id_new.groupby('batch_name').size()

batch_name
All_202606_00k-02k        60
All_202606_02k-04k        26
All_202606_04k-06k        31
All_202606_06k-08k        24
All_202606_08k-10k        11
All_202606_10k-12k        15
Health_202606_00k-02k    142
Health_202606_02k-04k     12
Health_202606_04k-06k     12
Health_202606_06k-08k     12
Health_202606_08k-10k      6
Health_202606_10k-12k      8
dtype: int64

# Step 3: Create new Target Collaboration in batches of 50 new creators

In [17]:
# Set Default Values
message = "Hi {{user_name}}! \n\nWe'd love to have you as a Vitami affiliate. We make PMS Relief Gummies to help women get through their period with less discomfort. You'll get 20% commission plus a free sample for 1 TikTok video. Kindly accept this invite and request your sample if you're interested so we can ship it right away. Hoping to spread the word on better period care together! \u200d\u200d"
end_time = "2101132799"
products = [{
    "id": '1734810555690551128',
    "target_commission_rate":2000, 
    "shop_ads_commission_rate": 300
}]
seller_contact_info = {
    "email": 'admin@vitamigummies.com'
}
free_sample_rule = {
    'has_free_sample': True,
    'is_sample_approval_exempt': True
}

In [18]:
# Load prior runs' manifest so chunk numbering continues from where each batch_name last left off, instead of restarting at 01 every run.
manifest_df = pd.read_csv(TARGET_COLLAB_MANIFEST_CSV)

## Batch process Target Collab

In [19]:
# create collab only if there are at N=optimize_cutoff in the group
optimize_collab_count = True
optimize_cutoff = 50

In [20]:
# Batch-create target collaborations: group by batch_name, then chunk each group's creators into groups of 50 (the API's max per invitation)
results = []

for batch_name, group in df_creators_list_id_new.groupby('batch_name'):
    creator_ids = group['creator_open_id'].tolist()
    chunks = [creator_ids[i:i + 50] for i in range(0, len(creator_ids), 50)]

    # Find the highest chunk_count already used for this batch_name in the manifest (across any previous run), and continue numbering from there.
    existing_counts = manifest_df.loc[manifest_df["batch_name"] == batch_name, "chunk_count"]
    start_count = int(existing_counts.max()) + 1 if not existing_counts.empty else 1

    for offset, chunk in enumerate(chunks):
        if optimize_collab_count:
            if len(chunk) < optimize_cutoff:
                continue
                
        chunk_count = start_count + offset
        name = f"{batch_name}_{chunk_count:02d}"
        print(f"Creating '{name}' with {len(chunk)} creators...")

        result = create_target_collaboration(
            name=name,
            end_time=end_time,
            products=products,
            creator_user_open_ids=chunk,
            seller_contact_info=seller_contact_info,
            free_sample_rule=free_sample_rule,
            message=message,
        )
        print(result)
        if result.get("code") == 0:
            collab_id = result["data"]["target_collaboration"]["id"]
            print(f"  ✅ Created: {collab_id}")
        else:
            collab_id = None
            print(f"  ⚠️  Failed: {result}")

        results.append({
            "name": name,
            "batch_name": batch_name,
            "chunk_count": chunk_count,
            "num_creators": len(chunk),
            "target_collaboration_id": str(collab_id),
            "code": result.get("code"),
            "message": result.get("message"),
        })

# Append a summary row per collaboration to the CSV manifest.
if results:
    df_results = pd.DataFrame(results)

    manifest_exists = Path(TARGET_COLLAB_MANIFEST_CSV).exists()
    df_results.to_csv(TARGET_COLLAB_MANIFEST_CSV, mode="a", header=not manifest_exists, index=False)

    print(f"\nAppended {len(df_results)} row(s) to: {TARGET_COLLAB_MANIFEST_CSV}")
else:
    print("\nNo creators to process this run.")

Creating 'All_202606_00k-02k_20' with 50 creators...
{'code': 0, 'data': {'target_collaboration': {'id': '7662397037648168724'}, 'target_collaboration_conflicts': []}, 'message': 'Success', 'request_id': '202607142256140789E195DE5E6701412D'}
  ✅ Created: 7662397037648168724
Creating 'Health_202606_00k-02k_26' with 50 creators...
{'code': 0, 'data': {'target_collaboration': {'id': '7662397263399601941'}, 'target_collaboration_conflicts': []}, 'message': 'Success', 'request_id': '202607142256155F2C1EC2F71870008ECB'}
  ✅ Created: 7662397263399601941
Creating 'Health_202606_00k-02k_27' with 50 creators...
{'code': 0, 'data': {'target_collaboration': {'id': '7662397032824899348'}, 'target_collaboration_conflicts': []}, 'message': 'Success', 'request_id': '20260714225616D7156EC40169DA008B3A'}
  ✅ Created: 7662397032824899348

Appended 3 row(s) to: target_collaboration_manifest.csv


In [21]:
manifest_df = pd.read_csv(TARGET_COLLAB_MANIFEST_CSV)

In [22]:
manifest_df['num_creators'].sum()

np.int64(2650)

In [23]:
manifest_df

,name,batch_name,chunk_count,num_creators,target_collaboration_id,code,message
0,All_202606_02k-04k_01,All_202606_02k-04k,1,21,7661970332203550472,0,Success
1,All_202606_04k-06k_01,All_202606_04k-06k,1,10,7661970508745393940,0,Success
2,All_202606_06k-08k_01,All_202606_06k-08k,1,12,7661970369088603924,0,Success
3,All_202606_08k-10k_01,All_202606_08k-10k,1,7,7661970518602401552,0,Success
4,All_202606_10k-12k_01,All_202606_10k-12k,1,2,7661970495009081108,0,Success
...,...,...,...,...,...,...,...
56,Health_202606_02k-04k_04,Health_202606_02k-04k,4,50,7662303241003337490,0,Success
57,Health_202606_02k-04k_05,Health_202606_02k-04k,5,50,7662303280861824784,0,Success
58,All_202606_00k-02k_20,All_202606_00k-02k,20,50,7662397037648168724,0,Success
59,Health_202606_00k-02k_26,Health_202606_00k-02k,26,50,7662397263399601941,0,Success
